# Jobs: temp stpop

project = ```iP-VAE```, host = ```mach```, device = ```any```

**Motivation**: <br>


In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_iclr_ipvae'))
from figures.convergence import plot_convergence
from main.config_defaults import default_configs
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = 'Dropbox/git/_iclr_ipvae/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'cleanup_chkpts.sh',
    'cleanup_recursive.sh',
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## mach

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'poisson'
t_train = 8

datasets = ['vH16-wht', 'MNIST']
temp_stop_list = [0.05, 0.04, 0.03, 0.02, 0.01]

betas = [1, 2, 4, 8]

In [6]:
for kl_beta in betas:
    for temp_stop in temp_stop_list:
        for dataset in datasets:
            # get architecture string
            dec = 'mlp' if dataset == 'MNIST' else 'lin'
            archi = f"ngd|{dec}"
            # get arg
            arg = ' '.join([
                f"--t_train {t_train}",
                f"--kl_beta {kl_beta}",
                f"--temp_stop '{temp_stop}'",
                '--comment temperature',
                '--verbose',
                # '--dry_run',
            ])
            gpu_i = tot % torch.cuda.device_count()
            scripts[gpu_i].append(job_runner_script(
                device=gpu_i,
                dataset=dataset,
                model=model_type,
                archi=archi,
                args=arg,
                seed=0,
            ))
            tot += 1

In [7]:
print(tot)

40

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 10, 1: 10, 2: 10, 3: 10}

### Save

In [9]:
n_fits = 5

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_iclr_ipvae/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 8 --kl_beta 1 --temp_stop '0.05' --comment 
temperature --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 8 --kl_beta 1 --temp_stop '0.03' --comment 
temperature --verbose

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_iclr_ipvae/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 8 --kl_beta 1 --temp_stop '0.01' --comment 
temperature --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 8 --kl_beta 2 --temp_stop '0.04' --comment 
temperature --verbose

[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_iclr_ipvae/scripts

...

./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 2 --temp_stop '0.01' --comment 
temperature --verbose && 
./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 4 --temp_stop '0.04' --comment 
temperature --verbose

[PROGRESS] 'mach-cuda3-fit3.txt' saved at
/home/hadi/Dropbox/git/_iclr_ipvae/scripts

./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 4 --temp_stop '0.02' --comment 
temperature --verbose && 
./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 8 --temp_stop '0.05' --comment 
temperature --verbose

[PROGRESS] 'mach-cuda3-fit4.txt' saved at
/home/hadi/Dropbox/git/_iclr_ipvae/scripts

./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 8 --temp_stop '0.03' --comment 
temperature --verbose && 
./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 8 --temp_stop '0.01' --comment 
temperature --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 8 --temp_stop '0.03' --comment 
temperature --verbose && 
./fit_model.sh '3' 'MNIST' 'poisson' 'ngd|mlp' --seed 0 --t_train 8 --kl_beta 8 --temp_stop '0.01' --comment 
temperature --verbose